# Cubic Equations of State

Computing compressibility factors and molar volumes using the Peng-Robinson and Soave-Redlich-Kwong cubic equations of state for pure methane and a methane-ethane mixture.

## Setup

Import the equation of state blocks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pathsim_chem.thermodynamics import PengRobinson, RedlichKwongSoave

## Pure Methane

The Peng-Robinson EoS solves a cubic equation in the compressibility factor $Z = Pv/(RT)$:

$$Z^3 - (1-B)Z^2 + (A - 3B^2 - 2B)Z - (AB - B^2 - B^3) = 0$$

where $A$ and $B$ are dimensionless parameters computed from the critical properties. Each block takes $(T, P)$ as inputs and returns $(v, Z)$.

In [ ]:
# Critical properties of methane
Tc_CH4 = 190.6    # K
Pc_CH4 = 4.6e6    # Pa
omega_CH4 = 0.011  # acentric factor

# Create EoS blocks
pr = PengRobinson(Tc=Tc_CH4, Pc=Pc_CH4, omega=omega_CH4)
rks = RedlichKwongSoave(Tc=Tc_CH4, Pc=Pc_CH4, omega=omega_CH4)

def eval_eos(block, T, P):
    """Evaluate an EoS block and return (v, Z)."""
    block.inputs[0] = T
    block.inputs[1] = P
    block.update(None)
    return block.outputs[0], block.outputs[1]

# Evaluate at 300 K, 1 atm
v_pr, Z_pr = eval_eos(pr, 300, 101325)
v_rks, Z_rks = eval_eos(rks, 300, 101325)

print(f"Methane at 300 K, 1 atm:")
print(f"  PR:  Z = {Z_pr:.6f}, v = {v_pr:.6e} m³/mol")
print(f"  RKS: Z = {Z_rks:.6f}, v = {v_rks:.6e} m³/mol")
print(f"  Ideal gas: Z = 1.0, v = {8.314462 * 300 / 101325:.6e} m³/mol")

## Compressibility Factor vs Pressure

At low pressure, $Z \to 1$ (ideal gas). As pressure increases, intermolecular forces cause $Z$ to deviate. At very high pressures, repulsive forces dominate and $Z > 1$.

In [ ]:
T_fixed = 250  # K (above Tc for methane, supercritical)
P_range = np.logspace(4, 7.5, 100)  # 10 kPa to ~30 MPa

Z_pr_arr = []
Z_rks_arr = []

for P in P_range:
    _, Z = eval_eos(pr, T_fixed, P)
    Z_pr_arr.append(Z)
    _, Z = eval_eos(rks, T_fixed, P)
    Z_rks_arr.append(Z)

fig, ax = plt.subplots(figsize=(7, 5))
ax.semilogx(P_range / 1e6, Z_pr_arr, label="Peng-Robinson")
ax.semilogx(P_range / 1e6, Z_rks_arr, "--", label="Soave-Redlich-Kwong")
ax.axhline(1.0, color="gray", linestyle="-.", alpha=0.5, label="Ideal gas")
ax.set_xlabel("Pressure [MPa]")
ax.set_ylabel("Compressibility Factor Z")
ax.set_title(f"Methane at T = {T_fixed} K")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Both EoS give very similar results. The characteristic dip below $Z = 1$ at moderate pressures reflects attractive intermolecular forces, while the rise above $Z = 1$ at high pressures is due to repulsive (excluded volume) effects.

## Z vs Reduced Temperature

Vary the temperature at a fixed pressure to see how the compressibility factor changes with reduced temperature $T_r = T / T_c$.

In [ ]:
P_fixed = 5e6  # 5 MPa
T_range = np.linspace(200, 600, 100)
Tr_range = T_range / Tc_CH4

Z_vs_T = []
for T in T_range:
    _, Z = eval_eos(pr, T, P_fixed)
    Z_vs_T.append(Z)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(Tr_range, Z_vs_T)
ax.axhline(1.0, color="gray", linestyle="-.", alpha=0.5)
ax.axvline(1.0, color="red", linestyle=":", alpha=0.5, label=r"$T_r = 1$ (critical)")
ax.set_xlabel(r"Reduced Temperature $T_r = T / T_c$")
ax.set_ylabel("Compressibility Factor Z")
ax.set_title(f"Peng-Robinson: Methane at P = {P_fixed/1e6:.0f} MPa")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Methane-Ethane Mixture

The EoS blocks support mixtures through standard van der Waals one-fluid mixing rules. Supply arrays of critical properties and mole fractions.

In [ ]:
# Critical properties: methane, ethane
Tc = [190.6, 305.3]      # K
Pc = [4.6e6, 4.872e6]    # Pa
omega = [0.011, 0.099]   # acentric factors

# Compare Z at different compositions
x1_range = np.linspace(0, 1, 20)  # methane mole fraction
T_mix, P_mix = 300, 3e6  # 300 K, 3 MPa

Z_mix = []
for x1 in x1_range:
    pr_mix = PengRobinson(
        Tc=Tc, Pc=Pc, omega=omega,
        x=[x1, 1 - x1],
    )
    _, Z = eval_eos(pr_mix, T_mix, P_mix)
    Z_mix.append(Z)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(x1_range, Z_mix, "o-")
ax.set_xlabel(r"$x_{\mathrm{CH_4}}$")
ax.set_ylabel("Compressibility Factor Z")
ax.set_title(f"PR: Methane-Ethane at T = {T_mix} K, P = {P_mix/1e6:.0f} MPa")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

The mixture compressibility factor varies smoothly with composition. Pure ethane (right side) has a lower $Z$ at these conditions because it is closer to its critical point ($T_c = 305.3$ K) and therefore deviates more from ideal gas behavior.